# MTPL frequency: scratch work

Explore source data and disposable feature ideas here. This notebook never updates the governed model-frame handoff; move accepted transforms into `01_data_ingestion.ipynb`. It can also open a published RAW candidate in SuperGLM's editor and export all categorical collapses as actual `LevelGrouping` objects for notebook 02.

In [ ]:
DATABASE_MODE = "local"
RUNTIME_MODULE = None
EXPECTED_REMOTE_DATABASE = ""
ALLOW_REMOTE_WRITES = False
REFRESH_LOCAL_RAW = False
MODEL_NAME = "MTPL_FREQ"
MODEL_LABEL = "Motor frequency"
DEPLOYMENT_SLOT = "MTPL_FREQ_UAT"
GROUPING_SOURCE_PACKAGE_VERSION = None  # None selects the latest published RAW package.
REPLACE_GROUPING_ARTIFACT = False

In [ ]:
from pathlib import Path
import sys

search_root = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        root
        for root in (search_root, *search_root.parents)
        if (root / "pricing_pipeline").is_dir()
        and (root / "pricing_models").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from inside the pricing repository.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from superglm.editor import EditorSession  # noqa: E402
from sqlalchemy import text  # noqa: E402

from pricing_pipeline.data.fremtpl import load_fremtpl_raw  # noqa: E402
from pricing_pipeline.infra.schema import schema_names_from_connectable  # noqa: E402
from pricing_pipeline.notebook import (  # noqa: E402
    connect,
    export_level_groupings,
    list_candidate_versions,
    load_registered_model,
    open_candidate,
)

MODEL_DIR = PROJECT_ROOT / "pricing_models/mtpl_frequency"
GROUPING_ARTIFACT_PATH = MODEL_DIR / ".local" / "routine_groupings.joblib"

In [ ]:
pricing = connect(
    mode=DATABASE_MODE,
    runtime_module=RUNTIME_MODULE,
    local_root=MODEL_DIR / ".local",
    expected_remote_database=EXPECTED_REMOTE_DATABASE,
    allow_remote_writes=ALLOW_REMOTE_WRITES,
)
display(pricing.destination)

## Read a source sample

In [ ]:
if pricing.mode == "local":
    load_fremtpl_raw(pricing.engine, replace=REFRESH_LOCAL_RAW)
schemas = schema_names_from_connectable(pricing.engine)
scratch = pd.read_sql_query(
    text(f"SELECT * FROM {schemas.pricing}.FREMTPL_RAW ORDER BY IDpol"),
    pricing.engine,
)
display(scratch.head())

## Disposable feature experiments

In [ ]:
scratch["CandidateLogDensity"] = np.log(scratch["Density"].clip(lower=1.0))
scratch.groupby("Area")["CandidateLogDensity"].describe()

## Optional: create the routine level-grouping artifact

This section is read-only in SQL but requires `DATABASE_MODE = "remote"`, because editable candidate bundles come from the remote workbench. Select levels in the widget and use **Collapse and refit** repeatedly across any categorical features you need.

In [ ]:
model = load_registered_model(
    pricing,
    model_name=MODEL_NAME,
    model_label=MODEL_LABEL,
    deployment_slot=DEPLOYMENT_SLOT,
    source_root=MODEL_DIR,
)
versions = list_candidate_versions(pricing, model=model)
raw_versions = versions.loc[versions["Kind"].eq("RAW")].copy()
display(raw_versions)
if raw_versions.empty:
    raise LookupError("No published RAW candidate is available for grouping.")
selected_package_version = (
    int(raw_versions.iloc[0]["Package"])
    if GROUPING_SOURCE_PACKAGE_VERSION is None
    else int(GROUPING_SOURCE_PACKAGE_VERSION)
)
if selected_package_version not in set(raw_versions["Package"].astype(int)):
    raise ValueError(
        "GROUPING_SOURCE_PACKAGE_VERSION is not in the displayed RAW list."
    )
grouping_candidate = open_candidate(
    pricing,
    model=model,
    package_version=selected_package_version,
)

In [ ]:
grouping_session = EditorSession.from_model(
    grouping_candidate.bundle.fitted_model,
    train_data=(
        grouping_candidate.bundle.X,
        grouping_candidate.bundle.y,
        grouping_candidate.bundle.sample_weight,
        grouping_candidate.bundle.offset,
    ),
    cv_report=grouping_candidate.bundle.cv_report,
)
display(grouping_session.widget())

## Export all current groupings

Run this after every intended collapse/refit has completed. The binary contains the real Python objects; the generated JSON sidecar is integrity and lineage evidence, not an analyst-edited configuration.

In [ ]:
grouping_artifact = export_level_groupings(
    grouping_candidate,
    editor_session=grouping_session,
    path=GROUPING_ARTIFACT_PATH,
    replace=REPLACE_GROUPING_ARTIFACT,
)
display(grouping_artifact)